# 00_b4 — Hungary: municipality population by age

Part of the `00_b*` series that replaces the five legacy population tables in the local
`dbis` Postgres (read by `traintran/population.py` / `harmonize_population_lau2.ipynb`) with
newer national-statistics files, each brought to the **same age-band widths** and keyed on
`LAU CODE` from `EU-27-LAU-2025-NUTS-2024.xlsx`.

> **Working notebook — not for publication.** It connects to the local DB only to take the legacy
> table as a baseline; once the harmonised output exists, nothing downstream needs the DB.

| | dataset | location |
|---|---|---|
| legacy (DB, census 2022) | `public.hu_population_lau2_21` | local `dbis` Postgres |
| legacy (DB) | `mdt.hungarylau2` | local `dbis` Postgres |
| new (2025) | KSH STATINFO `TA2025_W` — *Éves településstatisztikai adatok 2025-ös településszerkezetben*, indicators `TAAA101/107/111/110`, 31 Dec 2025, by settlement (<https://statinfo.ksh.hu/Statinfo/themeSelector.jsp?lang=hu>) | `data/eu/ref-national-statistical-offices/hu/TA2025_W_TAAA101_107_111_110_2025_telepulesek.csv` |

**Legacy notes.** Bands 0–14 / 15–64 / 65+ only, no total. No LAU code in the table: join by (name, district) to `mdt.hungarylau2`, whose bigint HCSO code needs `.zfill(5)`. 10 villages have a suppressed band.

In [1]:
import os, sys

# Run from the project root so every relative path ('data/...', 'traintran/...') resolves the
# same whether the notebook is launched from notebooks/ or from the root.
_here = os.path.abspath('')
if os.path.basename(_here) == 'notebooks':
    os.chdir(os.path.dirname(_here))
sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

Working directory: E:\notes\api_zssk\project


In [2]:
import os
from pathlib import Path

import pandas as pd
# import sqlalchemy as sa

import traintran  # noqa: F401  (fixes PROJ on import)
from traintran import population as pop
from traintran.config import default_config

COUNTRY = 'HU'
OUT_DIR = Path('data/eu/ref-national-statistical-offices') / COUNTRY.lower()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = default_config()
# Database comparison disabled. The 2025 population comes from the statistical office's files below; the
# database held only the older 2021 tables used for a comparison that feeds no output. To re-enable it, set
# TRAINTRAN_PG_URL in the environment and uncomment these lines and the cells marked '[disabled: ...]'.
# engine = sa.create_engine(os.environ['TRAINTRAN_PG_URL'])
print('Country   :', COUNTRY)
print('Output dir:', OUT_DIR)
# print('Postgres  :', cfg.pg_url.rsplit('@', 1)[-1])

Country   : HU
Output dir: data\eu\ref-national-statistical-offices\hu


## 1. Legacy — local DB

The raw table(s) as they sit in `dbis`, geometry columns dropped. A parquet snapshot is written
next to the new source (skip-if-present) so the comparison in §3 can later run without the DB.

In [3]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# GEOM_COLS = {'geom', 'geojson', 'wkb_geometry', 'geometry'}
# LEGACY_TABLES = ['public.hu_population_lau2_21', 'mdt.hungarylau2']


# def read_table(qualified):
#     schema, table = qualified.split('.', 1)
#     table = table.strip('"')
#     cols = [c['name'] for c in sa.inspect(engine).get_columns(table, schema=schema)
#             if c['name'].lower() not in GEOM_COLS]
#     sel = ', '.join('"' + c.replace('"', '""') + '"' for c in cols)
#     return pd.read_sql(f'select {sel} from {schema}."{table}"', engine)


# legacy_raw = {}
# for t in LEGACY_TABLES:
#     legacy_raw[t] = df = read_table(t)
#     print(f'{t}: {len(df):,} rows x {df.shape[1]} cols')
#     print('  ', list(df.columns))

In [4]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# for t, df in legacy_raw.items():
#     stem = t.replace('.', '_').replace(chr(34), '')
#     snap = OUT_DIR / f'legacy_db_{stem}.parquet'
#     if snap.exists():
#         print('[exists]', snap)
#     else:
#         df.to_parquet(snap, index=False)
#         print('[wrote] ', snap, f'({len(df):,} rows)')

### 1a. Baseline — the current loader on the legacy table

`pop.LOADERS['HU']` is exactly what `harmonize_population_lau2.ipynb` runs today; its keyed and
banded result is what the 2025 data has to reproduce in structure (same `lau_code` coverage,
same harmonised bands).

In [5]:
lau = pop.load_workbook_lau(cfg.lau_file, countries=(COUNTRY,))
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# legacy = pop.harmonise_bands(pop.LOADERS[COUNTRY](engine, lau), COUNTRY)
# legacy['country'] = COUNTRY
# legacy['ref_year'] = 2022

# print(pop.match_report({COUNTRY: legacy}, lau).to_string(index=False))
# print()
# print('band_method  :', legacy.band_method.unique().tolist())
# print('band_complete:', int(legacy.band_complete.sum()), '/', len(legacy))
# print('pop_total    :', f'{legacy.pop_total.sum():,.0f}')
# print()
# print(legacy.head().to_string())

## 2. New (2025) — KSH STATINFO, table `TA2025_W`

The KSH open-data API (<https://data.ksh.hu/openapi/>) only carries the EU High-Value Datasets,
whose finest geography is NUTS3. Settlement-level data is in the **STATINFO dissemination
database** (<https://statinfo.ksh.hu/Statinfo/themeSelector.jsp?lang=hu>), table `TA2025_W`, *Éves településstatisztikai adatok 2025-ös
településszerkezetben* (theme *Települési, járási statisztika*, code `CBBA`).

| indicator | meaning (from the catalogue, <https://www.ksh.hu/katalogus/tstar_bpstar_2025_katalogus.xlsx>) |
|---|---|
| `TAAA101` | resident population at year end, carried forward from the 2022 census |
| `TAAA107` | of which aged 0–14 |
| `TAAA111` | of which aged 15–64 |
| `TAAA110` | of which aged 65 and over |

Area level *Települések 2025.01.01.T-Star/TA* (all settlements), period 2025.

STATINFO is a session-based web application, not a REST API. `hu/ksh_statinfo_ta2025w_export.py`
replays the UI clicks in one session and saves the CSV export (windows-1250); its docstring lists
each click and the request it sends. It skips if the CSV is present (`--force` to refresh).

In [6]:
import subprocess

NEW_FILE = OUT_DIR / 'TA2025_W_TAAA101_107_111_110_2025_telepulesek.csv'
EXPORT_SCRIPT = OUT_DIR / 'ksh_statinfo_ta2025w_export.py'

if NEW_FILE.exists():
    print('[exists]', NEW_FILE, f'({NEW_FILE.stat().st_size / 1e6:.1f} MB)')
else:
    print('[download] running', EXPORT_SCRIPT.name)
    subprocess.run([sys.executable, str(EXPORT_SCRIPT), '--out', str(NEW_FILE)], check=True)

[exists] data\eu\ref-national-statistical-offices\hu\TA2025_W_TAAA101_107_111_110_2025_telepulesek.csv (2.6 MB)


### 2a. Raw export

Six title/blank lines, then one header row: `Terület` (settlement name), the four indicators, and
KSH's grouping attributes (*Ismérvek*): district, county, NUTS 2016, legal status, the T-STAR
settlement code (`Települések`, 4 digits, *not* the HCSO code), … Values use a decimal comma
(`"6055,0"`). 21 rows are non-settlements (*… területre nem bontható adatai*, *Országhatáron
kívüli tevékenység*) with blank values.

In [7]:
hu_raw = pd.read_csv(NEW_FILE, sep=';', skiprows=6, encoding='cp1250', dtype=str)
hu_raw = hu_raw.loc[:, ~hu_raw.columns.str.startswith('Unnamed')]
print(f'{len(hu_raw):,} rows x {hu_raw.shape[1]} cols')
print(list(hu_raw.columns))
hu_raw.head(3)

3,176 rows x 28 cols
['Terület', 'Lakónépesség száma az év végén (a népszámlálás végleges adataiból továbbvezetett adat) (fő)', 'Lakónépességből a 0-14 évesek száma az év végén (fő)', 'Lakónépességből a 15-64 évesek száma az év végén (fő)', 'Lakónépességből a 65 éves és idősebbek száma az év végén (fő)', 'Borvidékek, 2019', 'Agglomeráción belüli települések besorolása, 2003', 'Statisztikai nagyrégió', 'Agglomeráció típusa, 2003', 'Járások', 'Agglomerációk, 2003', 'Megyék - általános területi beosztás', 'Települések jogállása - összevont verzió, 1994', 'Kedvezményezett települések  kódjának nómenklatúrája', 'Borrégiók, 2019', 'Kedvezményezett járások kódjának nómenklatúrája', 'Turisztikai régiók', 'Települések', 'Világörökségi helyszín védőövezete', 'Tervezési statisztikai régiók', 'Szabad vállalkozási zónák', 'Az EU NUTS rendszere 2016', 'Területfejlesztési szempontból kiemelt térségek', 'Települések megyeszékhely jellege, 1991', 'Járási székhely', 'Települések jogállása - bővített ver

,Terület,Lakónépesség száma az év végén (a népszámlálás végleges adataiból továbbvezetett adat) (fő),Lakónépességből a 0-14 évesek száma az év végén (fő),Lakónépességből a 15-64 évesek száma az év végén (fő),Lakónépességből a 65 éves és idősebbek száma az év végén (fő),"Borvidékek, 2019","Agglomeráción belüli települések besorolása, 2003",Statisztikai nagyrégió,"Agglomeráció típusa, 2003",Járások,...,Világörökségi helyszín védőövezete,Tervezési statisztikai régiók,Szabad vállalkozási zónák,Az EU NUTS rendszere 2016,Területfejlesztési szempontból kiemelt térségek,"Települések megyeszékhely jellege, 1991",Járási székhely,"Települések jogállása - bővített verzió, 2005",Települések lakónépesség-nagyságkategóriái,Nemzeti parkok
0,Budapest,"1675777,0","201978,0","1124549,0","349250,0",9999 - Nem besorolható,1 - Központ,1 - Közép-Magyarország,0 - Budapesti agglomeráció,"999 - Budapest, illetve fiktív területi egység",...,99 - Nem világörökségi helyszín védőövezete,11 - Budapest (Budapest),9 - Nem szabad vállakozási zóna,HU110 - Budapest (2),4 - Budapesti Agglomeráció,1 - Megyeszékhely,"9 - Budapest, fiktív területi egység",10 - Főváros,10 - Budapest,99 - Nem tartozik nemzeti parkhoz
1,Aszód,"6055,0","906,0","4059,0","1090,0",9991 - A település nem tartozik borvidékbe - É...,9 - Településegyüttesbe nem tartozó település,1 - Közép-Magyarország,9 - Településegyüttesbe nem tartozó települések,122 - Aszódi járás,...,99 - Nem világörökségi helyszín védőövezete,12 - Pest (Pest),9 - Nem szabad vállakozási zóna,HU120 - Pest (2),9 - Nem tartozik területfejlesztési szempontbó...,2 - Nem megyeszékhely,1 - Székhelytelepülés,30 - Város,05 - 5000 - 9999,99 - Nem tartozik nemzeti parkhoz
2,Bag,"3696,0","590,0","2359,0","747,0",9991 - A település nem tartozik borvidékbe - É...,9 - Településegyüttesbe nem tartozó település,1 - Közép-Magyarország,9 - Településegyüttesbe nem tartozó települések,122 - Aszódi járás,...,99 - Nem világörökségi helyszín védőövezete,12 - Pest (Pest),9 - Nem szabad vállakozási zóna,HU120 - Pest (2),9 - Nem tartozik területfejlesztési szempontbó...,2 - Nem megyeszékhely,0 - Nem székhelytelepülés,41 - Nagyközség,04 - 2000 - 4999,99 - Nem tartozik nemzeti parkhoz


### 2b. Harmonise — one row per settlement

Names follow the legacy table (`municipality_name`, `district_name` without the *járás* suffix,
as in `hu_population_lau2_21`) and the series' band names (`popTotal`, `popAge0to14`,
`popAge15to64`, `popAge65over`). Kept attributes: county, NUTS3 (2016), T-STAR code, legal status.
Asserted: only type-90 non-settlement rows lack a total; values are whole persons; bands sum to
`popTotal`; names are unique.

**Blank band = zero.** 8 villages of 13–62 people have a blank 0–14 cell while the other two bands
already add up to the total (e.g. Nemesmedves 11 + 4 = 15), i.e. no children live there. A blank
band is filled with 0 only when the total minus the other bands is exactly 0; anything else fails.

In [8]:
IND = {'Lakónépesség száma az év végén (a népszámlálás végleges adataiból továbbvezetett adat) (fő)': 'popTotal',
       'Lakónépességből a 0-14 évesek száma az év végén (fő)': 'popAge0to14',
       'Lakónépességből a 15-64 évesek száma az év végén (fő)': 'popAge15to64',
       'Lakónépességből a 65 éves és idősebbek száma az év végén (fő)': 'popAge65over'}
STATUS = 'Települések jogállása - bővített verzió, 2005'
assert set(IND) <= set(hu_raw.columns), set(IND) - set(hu_raw.columns)

h = hu_raw.rename(columns={'Terület': 'municipality_name', **IND}).copy()
h['municipality_name'] = h['municipality_name'].str.strip()
for c in IND.values():
    h[c] = pd.to_numeric(h[c].str.strip().str.replace(',', '.', regex=False), errors='coerce')

blank = h['popTotal'].isna()
assert h.loc[blank, STATUS].str.startswith('90 -').all(), 'a real settlement has no value'
non_settlements = h.loc[blank, 'municipality_name'].tolist()
h = h[~blank].copy()

part = lambda s: s.str.split(' - ', n=1)
hu_new = pd.DataFrame({
    'municipality_name': h['municipality_name'],
    'district_name': part(h['Járások']).str[1].str.replace(r' járás$', '', regex=True),
    'county_name': part(h['Megyék - általános területi beosztás']).str[1],
    'nuts3_2016': part(h['Az EU NUTS rendszere 2016']).str[0],
    'tstar_code': part(h['Települések']).str[0],
    'legal_status': part(h[STATUS]).str[1],
    **{c: h[c] for c in IND.values()},
}).reset_index(drop=True)

B = ['popAge0to14', 'popAge15to64', 'popAge65over']
# A blank band is a zero only if the other bands already make up the total.
blank_band = hu_new[B].isna()
rest = hu_new.popTotal - hu_new[B].fillna(0).sum(axis=1)
assert (blank_band.sum(axis=1) <= 1).all() and (rest[blank_band.any(axis=1)] == 0).all(), \
    'a blank band is not explained by the total'
zero_filled = hu_new.loc[blank_band.any(axis=1), ['municipality_name', 'popTotal', *B]]
hu_new[B] = hu_new[B].fillna(0)
assert (hu_new[list(IND.values())] % 1 == 0).all().all(), 'values are not whole persons'
hu_new = hu_new.astype({c: 'int64' for c in IND.values()})
assert (hu_new[B].sum(axis=1) == hu_new.popTotal).all(), 'bands do not add up'
assert hu_new['municipality_name'].is_unique, 'settlement names repeat'
hu_new['ref_year'] = 2025
hu_new['ref_date'] = '2025-12-31'

print(f'hu_new: {len(hu_new):,} settlements x {hu_new.shape[1]} cols ({len(non_settlements)} non-settlement rows dropped)')
print(f'population total : {hu_new.popTotal.sum():,}')
print('bands sum to popTotal: all rows; names unique')
print(f'\nblank band filled with 0 ({len(zero_filled)} settlements):')
print(zero_filled.to_string(index=False))
hu_new.head()

hu_new: 3,155 settlements x 12 cols (21 non-settlement rows dropped)
population total : 9,488,428
bands sum to popTotal: all rows; names unique

blank band filled with 0 (8 settlements):
municipality_name  popTotal  popAge0to14  popAge15to64  popAge65over
      Nemesmedves      15.0          NaN          11.0           4.0
      Pusztaapáti      18.0          NaN          12.0           6.0
        Bódvarákó      62.0          NaN          51.0          11.0
          Debréte      13.0          NaN           4.0           9.0
     Tornakápolna      24.0          NaN          15.0           9.0
       Keresztéte      16.0          NaN          11.0           5.0
          Perecse      16.0          NaN          10.0           6.0
         Abaújlak      61.0          NaN          41.0          20.0


,municipality_name,district_name,county_name,nuts3_2016,tstar_code,legal_status,popTotal,popAge0to14,popAge15to64,popAge65over,ref_year,ref_date
0,Budapest,"Budapest, illetve fiktív területi egység",Budapest,HU110,1357,Főváros,1675777,201978,1124549,349250,2025,2025-12-31
1,Aszód,Aszódi,Pest vármegye,HU120,1618,Város,6055,906,4059,1090,2025,2025-12-31
2,Bag,Aszódi,Pest vármegye,HU120,0913,Nagyközség,3696,590,2359,747,2025,2025-12-31
3,Domony,Aszódi,Pest vármegye,HU120,0480,Község,2216,294,1480,442,2025,2025-12-31
4,Galgahévíz,Aszódi,Pest vármegye,HU120,1950,Község,2524,418,1637,469,2025,2025-12-31


### 2c. Joinability — new names vs the workbook, cross-checked through the legacy route

The export has no HCSO code, but its settlement names are unique and spelled like the 2025
workbook's `LAU NAME NATIONAL`, so the join is **name → `lau_code`**, asserted one-to-one.
As an independent check the legacy route is replayed on the new table — (name, district) →
`mdt.hungarylau2` HCSO code → `.zfill(5)` (`pop.hu_key`) — and both routes must give the same code.

In [9]:
wb_by_name = lau.set_index('lau_name_national')['lau_code']
assert wb_by_name.index.is_unique, 'workbook names repeat'
hu_new_lau = hu_new.assign(lau_code=hu_new['municipality_name'].map(wb_by_name))
print(pop.match_report({COUNTRY: hu_new_lau}, lau).to_string(index=False))
print('\nnew settlements not in the 2025 workbook:')
print(hu_new_lau.loc[hu_new_lau.lau_code.isna(), ['municipality_name', 'district_name', 'popTotal']]
      .to_string(index=False) or '  none')
print('\nworkbook LAU codes not covered by the new file:')
print(lau.loc[~lau.lau_code.isin(hu_new_lau.lau_code), ['lau_code', 'lau_name_national']]
      .to_string(index=False) or '  none')

# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Independent check: the legacy (name, district) -> HCSO route.
# ref = legacy_raw['mdt.hungarylau2'][['Locality name', 'District name', 'HCSO code of the locality']]
# ref = ref.rename(columns={'Locality name': 'municipality_name', 'District name': 'district_name'})
# ref['lau_code_ref'] = [pop.hu_key(v) for v in ref['HCSO code of the locality']]
# x = hu_new_lau.merge(ref[['municipality_name', 'district_name', 'lau_code_ref']],
#                      on=['municipality_name', 'district_name'], how='left')
# miss = x.lau_code_ref.isna()
# by_name = ref.drop_duplicates('municipality_name', keep=False).set_index('municipality_name')['lau_code_ref']
# x.loc[miss, 'lau_code_ref'] = x.loc[miss, 'municipality_name'].map(by_name)
# agree = (x.lau_code == x.lau_code_ref)
# print(f'\nname route vs legacy (name, district) -> HCSO route: {agree.sum():,} / {len(x):,} agree'
#       f' ({int(miss.sum())} needed the name-only fallback, e.g. Budapest)')
# print(x.loc[~agree, ['municipality_name', 'district_name', 'lau_code', 'lau_code_ref']].to_string(index=False) or '  all agree')

country  source_rows  workbook_lau  matched  source_unmatched  lau_uncovered
     HU         3155          3155     3155                 0              0

new settlements not in the 2025 workbook:


Empty DataFrame
Columns: [municipality_name, district_name, popTotal]
Index: []

workbook LAU codes not covered by the new file:
Empty DataFrame
Columns: [lau_code, lau_name_national]
Index: []


In [10]:
# [disabled: compares with the 2021 population tables in PostgreSQL; feeds no output -- see the setup cell]
# # Same LAU, two vintages: legacy census 2022 -> new 31 Dec 2025.
# cmp = (legacy[['lau_code', 'pop_total', 'y_0_14', 'y_15_64', 'y_65plus', 'band_complete']]
#        .merge(hu_new_lau[['lau_code', 'municipality_name', 'popTotal', *B]], on='lau_code', how='outer', indicator=True))
# print(cmp['_merge'].value_counts().to_string())
# b = cmp[(cmp._merge == 'both') & cmp.band_complete].copy()
# b['diff_%'] = 100 * (b.popTotal - b.pop_total) / b.pop_total
# print(f'\ntotal (complete legacy rows)  legacy {b.pop_total.sum():,.0f}  ->  new {b.popTotal.sum():,.0f}')
# print(b['diff_%'].describe().round(2).to_string())
# print('\nband shares (%), legacy -> new:')
# for o, n in [('y_0_14', 'popAge0to14'), ('y_15_64', 'popAge15to64'), ('y_65plus', 'popAge65over')]:
#     print(f'  {o:<9} {100 * b[o].sum() / b.pop_total.sum():5.1f} -> {100 * b[n].sum() / b.popTotal.sum():5.1f}')
# print('\nlegacy rows with a suppressed band, now:')
# inc = cmp[(cmp._merge == 'both') & ~cmp.band_complete.astype(bool)]
# print(inc[['lau_code', 'municipality_name', 'y_0_14', 'y_15_64', 'y_65plus', 'popTotal', *B]].to_string(index=False))
# print('\nlargest relative changes:')
# print(b.reindex(b['diff_%'].abs().sort_values(ascending=False).index)
#        .head(10)[['lau_code', 'municipality_name', 'pop_total', 'popTotal', 'diff_%']].round(1).to_string(index=False))

### 2d. Territorial changes

`TA2025_W` uses the 1 Jan 2025 settlement structure, the workbook's vintage, so no merger or split
is expected. `HU_MERGERS` is the same mechanism as Austria, empty; the assert fails on any code
that does not match.

In [11]:
HU_MERGERS = {}   # new LAU code -> workbook (2025) LAU codes it replaces

hu_new_lau['replaces_lau'] = hu_new_lau['lau_code'].map(lambda c: ';'.join(HU_MERGERS.get(c, [])) or None)
wb = set(lau['lau_code'])
covered = set(hu_new_lau['lau_code'].dropna()) | {o for olds in HU_MERGERS.values() for o in olds}
unexplained_new = hu_new_lau.loc[~hu_new_lau.lau_code.isin(wb | set(HU_MERGERS)), 'municipality_name'].tolist()
unexplained_wb = sorted(wb - covered)
assert not unexplained_new and not unexplained_wb, (unexplained_new, unexplained_wb)
print(f'new rows            : {len(hu_new_lau):,}')
print(f'workbook LAU covered: {len(wb & covered):,} / {len(wb):,}  ({len(HU_MERGERS)} mergers)')

new rows            : 3,155
workbook LAU covered: 3,155 / 3,155  (0 mergers)


### 2e. Join to the v4atua LAU subset (GeoJSON)

Hungarian features only (`CNTR_CODE = 'HU'`); `GISCO_ID = 'HU_<LAU code>'`.

In [12]:
import geopandas as gpd

LAU_SUBSET = Path('data/eu/ref-lau-2024-01m/subsetv4atua/LAU_RG_01M_2024_4326.geojson')
hu_geo = gpd.read_file(LAU_SUBSET, where="CNTR_CODE = 'HU'")
hu_geo['lau_code'] = hu_geo['GISCO_ID'].str.split('_', n=1).str[1]
assert hu_geo['lau_code'].is_unique
print(f'{LAU_SUBSET.name}: {len(hu_geo):,} HU features')

jg = hu_geo.drop(columns='geometry').merge(
    hu_new_lau[['lau_code', 'municipality_name', 'district_name', 'popTotal']],
    on='lau_code', how='outer', indicator=True)
print('\n' + jg['_merge'].value_counts().to_string())
print('\ngeojson only:', jg.loc[jg._merge == 'left_only', 'lau_code'].tolist())
print('new file only:', jg.loc[jg._merge == 'right_only', ['lau_code', 'municipality_name']].values.tolist())

m = jg[(jg._merge == 'both') & (jg.POP_2024 > 0)].copy()
m['diff_%'] = 100 * (m.popTotal - m.POP_2024) / m.POP_2024
print(f'\npopulation  GISCO POP_2024 {jg.POP_2024.sum():,.0f}  vs  new popTotal {jg.popTotal.sum():,.0f}')
print(m['diff_%'].describe().round(2).to_string())
name_diff = m[m.LAU_NAME.str.strip() != m.municipality_name.str.strip()]
print(f'\nsame code, different name: {len(name_diff)}')
print(name_diff[['lau_code', 'LAU_NAME', 'municipality_name']].head(15).to_string(index=False))
print('\nlargest relative changes:')
print(m.reindex(m['diff_%'].abs().sort_values(ascending=False).index)
       .head(8)[['lau_code', 'LAU_NAME', 'POP_2024', 'popTotal', 'diff_%']].round(1).to_string(index=False))

LAU_RG_01M_2024_4326.geojson: 3,155 HU features

_merge
both          3155
left_only        0
right_only       0

geojson only: []
new file only: []

population  GISCO POP_2024 9,584,627  vs  new popTotal 9,488,428
count    3155.00
mean       -1.13
std         4.58
min       -40.62
25%        -3.09
50%        -1.04
75%         0.83
max        40.48

same code, different name: 0
Empty DataFrame
Columns: [lau_code, LAU_NAME, municipality_name]
Index: []

largest relative changes:
lau_code    LAU_NAME  POP_2024  popTotal  diff_%
   23418        Sima      32.0        19   -40.6
   05351        Kára      42.0        59    40.5
   04987      Megyer      30.0        19   -36.7
   17066  Keresztéte      25.0        16   -36.0
   31556 Nemesmedves      23.0        15   -34.8
   03090     Borgáta     228.0       302    32.5
   18494       Garáb      50.0        34   -32.0
   29902   Bakonyság      55.0        39   -29.1


## 3. Write — common-schema dump

One CSV per country, same columns in all five `00_b*` notebooks, read by
`00_extract_base_data_eu.ipynb` §2c to attach the 2025 population to the LAU subset GeoJSON.

| column | content |
|---|---|
| `lau_code`, `gisco_id` | LAU code (text) and `<CC>_<lau_code>`, the subset key |
| `country`, `lau_name` | country code, national name |
| `pop_total`, `y_0_14`, `y_15_64`, `y_65plus` | persons (`pop.HARMONISED_BANDS`) |
| `band_method`, `ref_year`, `ref_date`, `source` | `exact`; reference year/date; dataset id |
| `native_id` | the source's own key (HU: KSH T-STAR settlement code) |
| `replaces_lau` | workbook LAU codes a merged row covers (`;`-joined), else empty |
| `includes_native` | native units summed into the row (`;`-joined), else empty |

Skips if the file exists; set `REWRITE_DUMP = True` to overwrite. Read it back with `dtype=str`
for the code columns (HU codes have leading zeros).

In [13]:
REWRITE_DUMP = False
DUMP_COLS = ['lau_code', 'gisco_id', 'country', 'lau_name', 'pop_total', *pop.HARMONISED_BANDS,
             'band_method', 'ref_year', 'ref_date', 'source', 'native_id', 'replaces_lau', 'includes_native']
DUMP_FILE = OUT_DIR / f'lau_population_2025_{COUNTRY.lower()}.csv'

f = hu_new_lau.reset_index(drop=True)
dump = pd.DataFrame({
    'lau_code': f['lau_code'].astype(str),
    'gisco_id': COUNTRY + '_' + f['lau_code'].astype(str),
    'country': COUNTRY,
    'lau_name': f['municipality_name'],
    'pop_total': f['popTotal'], 'y_0_14': f['popAge0to14'], 'y_15_64': f['popAge15to64'], 'y_65plus': f['popAge65over'],
    'band_method': 'exact',
    'ref_year': f['ref_year'], 'ref_date': f['ref_date'],
    'source': 'KSH_TA2025_W',
    'native_id': f['tstar_code'].astype(str),   # KSH T-STAR settlement code (4 digits, text)
    'replaces_lau': f['replaces_lau'],
    'includes_native': None,
})[DUMP_COLS]

assert dump['lau_code'].is_unique and dump['gisco_id'].is_unique
assert (dump[pop.HARMONISED_BANDS].sum(axis=1) == dump['pop_total']).all(), 'bands do not add up'
assert dump[['pop_total', *pop.HARMONISED_BANDS]].notna().all().all()
replaced = {o for r in dump['replaces_lau'].dropna() for o in r.split(';')}
merged_new = set(dump.loc[dump['replaces_lau'].notna(), 'lau_code'])
assert (set(dump['lau_code']) - merged_new) | replaced == set(lau['lau_code']), 'dump does not cover the workbook LAUs'

if DUMP_FILE.exists() and not REWRITE_DUMP:
    print('[exists]', DUMP_FILE, '-- set REWRITE_DUMP = True to overwrite')
else:
    dump.to_csv(DUMP_FILE, index=False, encoding='utf-8')
    print('[wrote] ', DUMP_FILE)
back = pd.read_csv(DUMP_FILE, dtype={'lau_code': str, 'gisco_id': str, 'native_id': str})
assert list(back.columns) == DUMP_COLS and back['lau_code'].tolist() == dump['lau_code'].tolist(), 'file differs from dump'
print(f'{len(back):,} rows, population {back.pop_total.sum():,}; '
      f"merged rows {int(back.replaces_lau.notna().sum())}, aggregated rows {int(back.includes_native.notna().sum())}")
back.head(3)

[exists] data\eu\ref-national-statistical-offices\hu\lau_population_2025_hu.csv -- set REWRITE_DUMP = True to overwrite
3,155 rows, population 9,488,428; merged rows 0, aggregated rows 0


,lau_code,gisco_id,country,lau_name,pop_total,y_0_14,y_15_64,y_65plus,band_method,ref_year,ref_date,source,native_id,replaces_lau,includes_native
0,13578,HU_13578,HU,Budapest,1675777,201978,1124549,349250,exact,2025,2025-12-31,KSH_TA2025_W,1357,NaN,NaN
1,16188,HU_16188,HU,Aszód,6055,906,4059,1090,exact,2025,2025-12-31,KSH_TA2025_W,1618,NaN,NaN
2,09131,HU_09131,HU,Bag,3696,590,2359,747,exact,2025,2025-12-31,KSH_TA2025_W,0913,NaN,NaN
